In [13]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate

from datasets import load_dataset
from utils.func import read_jsonl, softmax
from utils.metric import evaluate

model_name = "LLaVA-7B"
fix = ""
data = read_jsonl(f"./output/{model_name}/MathV{fix}.jsonl")
len(data)

0it [00:00, ?it/s]

1000

In [14]:
dataset = load_dataset("AI4Math/MathVista", split='testmini')
len(dataset)

1000

In [15]:
if not os.path.exists(f"./output/{model_name}/MathV{fix}_output.json"):
    res = {}
    for pid in range(len(dataset)):
        dic = dataset[pid]
        del dic['decoded_image']
        dic['response'] = data[pid]['response']
        res[pid+1] = dic

    json.dump(res, open(f"./output/{model_name}/MathV{fix}_output.json", 'w'))

In [16]:
print(f"""python extract_answer.py \\
    --output_dir "../../TowardsTrustworthy/output/{model_name}/" \\
    --output_file "MathV{fix}_output.json" \\
    --llm_engine "gpt-4-0125-preview" """)

python extract_answer.py \
    --output_dir "../../TowardsTrustworthy/output/LLaVA-7B/" \
    --output_file "MathV_output.json" \
    --llm_engine "gpt-4-0125-preview" 


In [17]:
print(f"""python calculate_score.py \\
    --output_dir "/data/qinyu/research/TowardsTrustworthy/output/{model_name}/" \\
    --output_file "MathV{fix}_output.json" """)

python calculate_score.py \
    --output_dir "/data/qinyu/research/TowardsTrustworthy/output/LLaVA-7B/" \
    --output_file "MathV_output.json" 


### Please evaluate the results use the codes provided by the MathVista repo
After that, you can run the following codes

In [18]:
data = json.load(open(f"./output/{model_name}/MathV{fix}_output.json"))
logits = read_jsonl(f"./output/{model_name}/MathV{fix}.jsonl")

0it [00:00, ?it/s]

In [22]:
data['1']

{'pid': '1',
 'question': "When a spring does work on an object, we cannot find the work by simply multiplying the spring force by the object's displacement. The reason is that there is no one value for the force-it changes. However, we can split the displacement up into an infinite number of tiny parts and then approximate the force in each as being constant. Integration sums the work done in all those parts. Here we use the generic result of the integration.\r\n\r\nIn Figure, a cumin canister of mass $m=0.40 \\mathrm{~kg}$ slides across a horizontal frictionless counter with speed $v=0.50 \\mathrm{~m} / \\mathrm{s}$. It then runs into and compresses a spring of spring constant $k=750 \\mathrm{~N} / \\mathrm{m}$. When the canister is momentarily stopped by the spring, by what distance $d$ is the spring compressed?",
 'image': 'images/1.jpg',
 'choices': None,
 'unit': None,
 'precision': 1.0,
 'answer': '1.2',
 'question_type': 'free_form',
 'answer_type': 'float',
 'metadata': {'cate

In [24]:
X = np.array([ins['logits'] for ins in logits])
# y = np.array([1 if data[str(i)]["true_false"] else 0 for i in range(1, 1001)])

y = np.array([
    1 if str(data[str(i)]["response"]).strip() == str(data[str(i)]["answer"]).strip() else 0
    for i in range(1, 1001)
])

In [25]:
model = LogisticRegression()
res = cross_validate(model, X, y, cv=10, scoring=('roc_auc', 'accuracy', 'f1'))
print(res['test_roc_auc'])
print(res['test_accuracy'])
print(res['test_f1'])

print(f"AUROC: {np.mean(res['test_roc_auc'])*100:.2f}")
print(f"ACC: {np.mean(res['test_accuracy'])*100:.2f}")
print(f"F1: {np.mean(res['test_f1'])*100:.2f}")

[0.76494565 0.81793478 0.9076087  0.8125     0.66576087 0.82661783
 0.76923077 0.66910867 0.87912088 0.7020757 ]
[0.88 0.88 0.91 0.9  0.87 0.93 0.91 0.88 0.93 0.89]
[0.25       0.14285714 0.30769231 0.16666667 0.23529412 0.53333333
 0.30769231 0.25       0.58823529 0.15384615]
AUROC: 78.15
ACC: 89.80
F1: 29.36


Please use the LVLMs to self-evaluate their solutions, and run the following codes

In [28]:
data = json.load(open(f"./output/{model_name}/MathV{fix}_output.json"))
logits = read_jsonl(f"./output/{model_name}/MathV{fix}_self_eval.jsonl")

X = np.array([ins['logits'] for ins in logits])
# y = np.array([1 if data[str(i)]["true_false"] else 0 for i in range(1, 1001)])
y = np.array([
    1 if str(data[str(i)]["response"]).strip() == str(data[str(i)]["answer"]).strip() else 0
    for i in range(1, 1001)
])

model = LogisticRegression()
res = cross_validate(model, X, y, cv=10, scoring=('roc_auc', 'accuracy', 'f1'))
print(res['test_roc_auc'])
print(res['test_accuracy'])
print(res['test_f1'])

print(f"AUROC: {np.mean(res['test_roc_auc'])*100:.2f}")
print(f"ACC: {np.mean(res['test_accuracy'])*100:.2f}")
print(f"F1: {np.mean(res['test_f1'])*100:.2f}")

0it [00:00, ?it/s]

[0.76494565 0.81793478 0.9076087  0.8125     0.66576087 0.82661783
 0.76923077 0.66910867 0.87912088 0.7020757 ]
[0.88 0.88 0.91 0.9  0.87 0.93 0.91 0.88 0.93 0.89]
[0.25       0.14285714 0.30769231 0.16666667 0.23529412 0.53333333
 0.30769231 0.25       0.58823529 0.15384615]
AUROC: 78.15
ACC: 89.80
F1: 29.36
